### Background info

5 source of truth (SOT) files:

1) SDTM IG
 - CDISC status: finalized, publically reviewed, FINAL
 - file name: SDTM_and_SDTMIG_Conformance_Rules_v2.0 (8).xlsx
 - sheet name: SDTMIG Conformance Rules v2.0
 - Rule ID: CGXXXX

2) ADaM IG
 - CDISC status: finalized, publically reviewed, FINAL
 - file name: ADaM Conformance Rules v5.0 (3).xlsx
 - sheet name: Rules Catalogue
 - Rule ID: numerical, up to 3 digits

3) FDA Business Rules
 - CDISC status: finalized, not publically reviewed, Number of rules final but likely work to be done
 - file name: FDA_Business_Rules_Harmonized Rules Catalog Spreadsheet.xlsx
 - sheet name: Rules Catalog
 - Rule ID: FBXXXX

4) Define-XML conformance, probably schema rules 
 - CDISC status: finalized, publically reviewed, NOT FINAL
 - file name: Define-XML_v2.1_Conformance_Rules (6).xlsx
 - sheet name: DefineRules
 - Rule Identifier: numerical, up to 3 digits

5) Define-XML cross-checks, 
 - CDISC status: in active development as of today (2025-12-03)
 - file name: 360i Rules Catalog.xlsx
 - sheet name: Rules Catalog
 - Rule ID: author abbreviation


# New Process for Regression

1) SOT for conformance rules, encoded by conformance IDs, are in the above 5 Excel files in the "Rule Id" or equivalent column. However, it is actually not a rule ID, but a conformance ID, because each conformance ID can be implemented by 1-n rules. From the SOT files we get a list of conformance_ids, the count of which = the total number of conformance that must be implemented.
2) we read in each of the 5 files, and create a conformance_id column according to the following rules:
 - SDTM IG: we take the CGXXXX IDs
 - ADaM IG: we convert the numbers into a ADXXXX
 - FDA Business Rules: we take the FBXXXX IDs
 - Define Schema rules: we convert the numbers into DSXXXX
 - Define Cross Check rules: we create DCXXXX
3) we then concat the above into a single dataframe called concat_sot, where unique row is identified by the conformance_id
4) we then pull down the latest dump from the rule editor and merge as appropriate, such that the resulting unique ID per row is the CORE-ID, if assigned, or empty for those in draft, and there could be 1-n mappings from conformance_id to rule_id
5) we also merge the other DF on the sharepoint to get more metadata for rules, especially their classification into rules types

We then have an actual count of conformance_ids (total conformance to implement) and rule_ids (total rules available, and their status).

There is no way to automatically tell when a conformance_id has been fully implemented - this status needs to be updated manually by CDISC/us in the original 5 SOT files. I will add the column "all_implementations_present" as a flag to indicate that no more rules will have to be defined to cover the conformance_id. This flag will be counted to indicate overall progress. If, then, this flag is set and all rules associated with the conformance ID pass regression, the conformance_id is "complete" and ready for release.

## 0: Setup:

In [6]:
import pandas as pd
import os
import re
from datetime import datetime

dump_version = "2025-12-03"

## SOT-1: SDTM IG Conformance

In [7]:
sdtmig = pd.read_excel(f"{dump_version}-SOT-sources/SDTM_and_SDTMIG_Conformance_Rules_v2.0 (8).xlsx", sheet_name='SDTMIG Conformance Rules v2.0', skiprows=0)
sdtmig.columns = sdtmig.columns.str.replace(r'\s+', ' ', regex=True).str.strip()
sdtmig["conformance_id"] = sdtmig["Rule ID"]
# uncomment below line if conformance IDs do not get a new ID when logic changes across version standards
# sdtmig["conformance_id"] = sdtmig["conformance_id"] + "_" + sdtmig["SDTMIG Version"].astype(str)
sdtmig = sdtmig[["conformance_id"] + sdtmig.columns[:-1].tolist()]

# sort and take latest to track conformance rule
sdtmig = sdtmig.sort_values(by=["conformance_id", "SDTMIG Version"])
print(sdtmig.shape)
sdtmig = sdtmig.drop_duplicates(subset="conformance_id", keep="last")
print(sdtmig.shape)

# sanity check
sdtmig.head(5)

(1315, 14)
(534, 14)


,conformance_id,Rule ID,SDTMIG Version,Rule Version,Class,Domain,Variable,Condition,Rule,Document,Section,Item,Cited Guidance,Release Notes
2,CG0001,CG0001,3.4,1,ALL,ALL,DOMAIN,Not custom domain,DOMAIN = valid Domain Code published by CDISC,IG v3.4,3.2.2,NaN,Using SDTM-specified standard domain names and...,Section and cited guidance updated. Orginal c...
5,CG0002,CG0002,3.4,1,ALL,ALL,--DUR,--DUR ^= null,--DUR collected and not derived,Model v2.0,Timing,--DUR,Used only if collected on the CRF and not deri...,Cited guidance updated. Reference to model sec...
8,CG0006,CG0006,3.4,2,ALL,ALL,--DY,Date portion of --DTC is complete and date por...,--DY calculated as per the study day algorithm...,IG v3.4,4.4.4,NaN,The Study Day value is incremented by 1 for ea...,No Change
11,CG0007,CG0007,3.4,1,ALL,ALL,--DY,The date portion of --DTC ^= complete date or ...,--DY = null,IG v3.4,4.4.4,NaN,"The permissible Study Day variables (--DY, --S...",No Change
14,CG0008,CG0008,3.4,1,ALL,ALL,--ELTM,--TPTREF = null,--ELTM = null,Model v2.0,Timing,--ELTM,The interval of time between a planned time po...,Cited guidance updated. Reference to model sec...


## SOT-2: ADaM IG Conformance


In [8]:
adamig = pd.read_excel(f"{dump_version}-SOT-sources/ADaM Conformance Rules v5.0 (3).xlsx", sheet_name='Rules Catalogue', skiprows=1)
adamig.columns = adamig.columns.str.replace(r'\s+', ' ', regex=True).str.strip()
adamig["conformance_id"] = "AD" + adamig["Rule ID"].apply(lambda x: f"{int(x):04d}")
# uncomment below line if conformance IDs do not get a new ID when logic changes across version standards
# adamig["conformance_id"] = adamig["conformance_id"] + "_" + adamig["Rule Set (Generally IG Version, OCCDS v1.0, ADNCA v1.0)"].astype(str)
adamig = adamig[["conformance_id"] + adamig.columns[:-1].tolist()]

# sort and take latest to track conformance rule
adamig = adamig.sort_values(by=["conformance_id", "Rule Set (Generally IG Version, OCCDS v1.0, ADNCA v1.0)"])
print(adamig.shape)
adamig = adamig.drop_duplicates(subset="conformance_id", keep="last")
print(adamig.shape)

# sanity check
adamig.head(5)

(1963, 24)
(790, 24)


,conformance_id,Rule ID,Rule ID Version (represents any change to the rule),Related Rule(s),"Rule Set (Generally IG Version, OCCDS v1.0, ADNCA v1.0)",Class,Subclass,SEND/SDTM Domain,Variable or Item,Define-XML Element,...,Natural Language Rule (Failure Criteria),Rule (Failure Criteria),Condition (Failure),Rule Section,Implementation Guide (Cited document),Cited Section,Cited Item (text; figure; table; footnote),Cited Guidance,Guidance Section,Release Notes
3,AD0001,1,1,NaN,1.3,SUBJECT LEVEL ANALYSIS DATASET,NaN,NaN,NaN,NaN,...,ADSL dataset does not exist,ADSL dataset does not exist,NaN,NaN,Model v2.1; ADaMIG v1.3,6; 2.3.1,NaN,"Model v2.1, Section 6: ADSL and its related me...",NaN,NaN
7,AD0002,2,1,NaN,1.3,ALL,NaN,NaN,NaN,NaN,...,A variable is present in ADaM with the same na...,Variables do not have identical labels,A variable is present in ADaM with the same na...,NaN,Model v2.1; ADaMIG v1.3,4.1.2; 3.1.1,3,"Model v2.1, Section 4.1.2: Any ADaM variable w...",NaN,NaN
11,AD0005,5,1,NaN,1.3,ALL,NaN,NaN,*FL,NaN,...,A variable with a suffix of FL has a value tha...,"A variable has a value that is not Y, N or null",Variable with a suffix of FL,NaN,ADaMIG v1.3,3.1.4,4; 9,"ADaMIG v1.3, Section 3.1.4, Item 4: For subjec...",NaN,NaN
15,AD0006,6,1,NaN,1.3,ALL,NaN,NaN,*FN,NaN,...,A variable with a suffix of FL is present and ...,A variable with a suffix of FN has a value tha...,Variable with a suffix of FL is present and a ...,NaN,ADaMIG v1.3,3.1.4,5; 9,"ADaMIG v1.3, Section 3.1.4, Item 5: For subjec...",NaN,NaN
19,AD0007,7,1,NaN,1.3,ALL,NaN,NaN,*FL,NaN,...,A variable with a suffix of FN is present but ...,A variable is not present that has a suffix of...,Variable with a suffix of FN is present,NaN,ADaMIG v1.3,3.1.1,8,"ADaMIG v1.3, Section 3.1.1, Item 8: Variables ...",NaN,NaN


## SOT-3: FDA Business Rules Conformance

In [9]:
fdabr = pd.read_excel(f"{dump_version}-SOT-sources/FDA_Business_Rules_Harmonized Rules Catalog Spreadsheet.xlsx", sheet_name='Rules Catalog', skiprows=1)
fdabr.columns = fdabr.columns.str.replace(r'\s+', ' ', regex=True).str.strip()
fdabr["conformance_id"] = fdabr["Rule ID"]
fdabr = fdabr[["conformance_id"] + fdabr.columns[:-1].tolist()]
fdabr.head(5)

/Users/verisian/.pyenv/versions/verisian-core/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


,conformance_id,Rule ID,Rule ID Version (represents any change to the rule),"Related Rule(s) [See Also, Compare Against]","Rule Set (Generally IG Version, OCCDS v1.0, ADNCA v1.0)",Class,Subclass,Dataset or Domain,Variable,Element,...,Cited Document,Cited Section,"Cited Item (text, figure, table, footnote)","Cited Guidance (start with variable you are referring to if, for example it is from CDISC Notes)",Guidance Section,Release Notes,Author,Question(s) for FDA,Answer of the FDA,Authoring section
0,FB0101,FB0101,1,NaN,NaN,REL,NaN,SUPPAE,AETRTEM,NaN,...,FDA Business Rules v1.5,FDAB001,NaN,A treatment-emergent flag should be submitted.,NaN,NaN,Nick,NaN,NaN,NaN
1,FB0301,FB0301,1,NaN,NaN,EVT,NaN,AE,NaN,NaN,...,FDA Business Rules v1.5,FDAB003,NaN,Adverse events should be coded using the most ...,NaN,NaN,Anthony,NaN,NaN,NaN
2,FB0401,FB0401,1,NaN,NaN,SPC,NaN,SE,NaN,,...,FDA Business Rules v1.5,FDAB004,NaN,"AE, CE, CM, DS, EG, EX, LB, MH, PC, PP, SE, SV...",NaN,NaN,Mandar,NaN,NaN,NaN
3,FB0402,FB0402,1,NaN,NaN,EVT,NaN,AE,NaN,,...,FDA Business Rules v1.5,FDAB004,NaN,"AE, CE, CM, DS, EG, EX, LB, MH, PC, PP, SE, SV...",NaN,NaN,Mandar,NaN,NaN,NaN
4,FB0403,FB0403,1,NaN,NaN,EVT,NaN,DS,NaN,,...,FDA Business Rules v1.5,FDAB004,NaN,"AE, CE, CM, DS, EG, EX, LB, MH, PC, PP, SE, SV...",NaN,NaN,Mandar,NaN,NaN,NaN


## SOT-4: Define Schema Validation Conformance

In [10]:
defs = pd.read_excel(f"{dump_version}-SOT-sources/Define-XML_v2.1_Conformance_Rules (6).xlsx", sheet_name='DefineRules', skiprows=0)
defs.columns = defs.columns.str.replace(r'\s+', ' ', regex=True).str.strip()
defs["Rule Identifier"] = defs["Rule Identifier"].apply(lambda x: f"{int(x):04d}")
defs["conformance_id"] = "DS" + defs["Rule Identifier"]
defs = defs[["conformance_id"] + defs.columns[:-1].tolist()]
defs.head(5)

,conformance_id,Rule Identifier,Displayed Link,Plain Text Rule,Rule,Rule Message,Applicable Versions,Source Type,XPaths,Element,Attribute
0,DS0001,0001,"<a href=""<a href=""https://wiki.cdisc.org/displ...",Elements must be ordered in accordance with th...,//<element OID> is not allowed under [parent e...,Element [element]/@OID s not allowed under [pa...,Both 2.0 and 2.1,Schema,NaN,ALL elements,NaN
1,DS0002,0002,"<a href=""<a href=""https://wiki.cdisc.org/displ...",Element ODM must be provided in any Define-XML...,/ODM must be provided.,Element ODM is missing in the Define-XML docum...,Both 2.0 and 2.1,Specification,/ODM,ODM,NaN
2,DS0003,0003,"<a href=""<a href=""https://wiki.cdisc.org/displ...",No more than one ODM element may be provided i...,No more than one ODM element may be provided.,Element ODM is provided more than once in the ...,Both 2.0 and 2.1,Specification,/ODM,ODM,NaN
3,DS0004,0004,"<a href=""<a href=""https://wiki.cdisc.org/x/Z6O...",The value of attribute CreationDateTime must b...,The value of @CreationDateTime must be in ISO ...,The value of CreationDateTime[@CreationDateTim...,Both 2.0 and 2.1,Schema,/ODM,ODM,CreationDateTime
4,DS0005,0005,"<a href=""<a href=""https://wiki.cdisc.org/x/Z6O...",The value of attribute AsOfDateTime must be in...,The value of @AsOfDateTime must be in ISO 8601...,The value of AsOfDateTime[@AsOfDateTime] is no...,Both 2.0 and 2.1,Schema,/ODM,ODM,AsOfDateTime


## SOT-5: Define Cross Checks Validation Conformance

In [11]:
defc = pd.read_excel(f"{dump_version}-SOT-sources/360i Rules Catalog.xlsx", sheet_name='Rules Catalog', skiprows=1)
defc.columns = defc.columns.str.replace(r'\s+', ' ', regex=True).str.strip()
print(defc.shape)
defc = defc[defc["Type = Dataset, CLIB, OR Schema"].isin(["CLIB","Dataset"])].reset_index(drop=True).copy()
print(defc.shape)
defc["conformance_id"] = defc.index
defc["conformance_id"] = defc["conformance_id"].apply(lambda x: f"{int(x):04d}")
defc["conformance_id"] = "DC" + defc["conformance_id"]
defc = defc[["conformance_id"] + defc.columns[:-1].tolist()]
defc.head(5)

(297, 29)
(42, 29)


,conformance_id,Define Section,Rule ID,Rule ID Version (represents any change to the rule),"Related Rule(s) [See Also, Compare Against]","Rule Set (Generally IG Version, OCCDS v1.0, ADNCA v1.0)",Class,Subclass,Dataset or Domain,Variable,...,Executability,Rule Section,Cited Document,Cited Section,"Cited Item (text, figure, table, footnote)","Cited Guidance (start with variable you are referring to if, for example it is from CDISC Notes)",Guidance Section,Release Notes,Comments/questions for discussion,Response Lex
0,DC0000,Datasets,SM,NaN,NaN,SDTM,NaN,NaN,NaN,NaN,...,Fully executable,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,DC0001,Datasets,SM,NaN,NaN,SDTM,NaN,NaN,NaN,NaN,...,Fully executable,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,DC0002,Datasets,SM,NaN,NaN,SDTM\ADaM,NaN,NaN,NaN,NaN,...,Fully executable,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,DC0003,Datasets,SM,NaN,NaN,ADaM,NaN,NaN,NaN,NaN,...,Fully executable,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,DC0004,Datasets,SM,NaN,NaN,SDTM,NaN,NaN,NaN,NaN,...,Fully executable,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Concatenate into single SOT dataframe

In [12]:
def ensure_columns(df, cols):
    for col in cols:
        if col not in df.columns:
            df[col] = ""
    return df

In [13]:
#print("SDTM")
#print(sdtmig.columns)
sdtm_convert_dict = {"SDTMIG Version ":"cg_version"}
sdtmig = sdtmig.rename(columns=sdtm_convert_dict)

#print("ADAM")
#print(adamig.columns)
adamig_convert_dict = {"Rule Set (Generally IG Version, OCCDS v1.0, ADNCA v1.0)": "cg_version",
                       "Rule ID Version (represents any change to the rule)": "Rule Version",
                       "SEND/SDTM Domain": "Domain",
                       "Variable or Item": "Variable",
                       "Implementation Guide (Cited document)": "Document",
                       "Cited Section": "Section",
                       "Cited Item (text; figure; table; footnote)": "Item"
                       }
adamig = adamig.rename(columns=adamig_convert_dict)

#print("FDA")
#print(fdabr.columns)
fdabr_convert_dict = {"Rule Set (Generally IG Version, OCCDS v1.0, ADNCA v1.0)":"cg_version",
                      "Rule ID Version (represents any change to the rule)": "Rule Version",
                      "Dataset or Domain": "Domain",
                      "Element": "Define-XML Element",
                      "Cited Document": "Document",
                      "Cited Section": "Section",
                      "Cited Item (text, figure, table, footnote)": "Item",
                      "Cited Guidance (start with variable you are referring to if, for example it is from CDISC Notes)": "Cited Guidance"
                      }
fdabr = fdabr.rename(columns=fdabr_convert_dict)

#print("DEFS")
#print(defs.columns)
defs_convert_dict = {"Rule Identifier":"Rule ID", "Applicable Versions":"cg_version",
                     "Rule Message": "Error Message",
                     "Source Type": "Define Source Type",
                     "Element": "Define XML Element",
                     "Attribute": "Define XML Attribute",
                     }
defs = defs.rename(columns=defs_convert_dict)

#print("DEFC")
#print(defc.columns)
defc_convert_dict = {"Rule Set (Generally IG Version, OCCDS v1.0, ADNCA v1.0)":"cg_version",
                     "Rule ID Version (represents any change to the rule)": "Rule Version",
                     "Dataset or Domain": "Domain",
                     "Element": "Define XML Element",
                     "Type = Dataset, CLIB, OR Schema": "Define-XML Rule Type",
                     "Cited Document": "Document",
                     "Cited Section": "Section",
                     "Cited Item (text; figure; table; footnote)": "Item",
                     "Cited Guidance (start with variable you are referring to if, for example it is from CDISC Notes)": "Cited Guidance"
                     }
defc = defc.rename(columns=defc_convert_dict)


target_columns = ["conformance_id", "Rule ID", "cg_version", "Rule Version", "Class", "Subclass", "Domain", "Variable", "Define-XML Element", "Scope Section",

                  "Natural Language Rule (Success Criteria)", "Rule (Success Criteria)", "Condition (Success)", "Natural Language Rule (Failure Criteria)", "Rule (Failure Criteria)", "Condition (Failure)", "Rule Section", "Condition" ,

                  "Plain Text Rule", "Rule", "Error Message",

                  "Define-XML Rule Type", "Define Section", "Define Source Type", "Define XML Attribute", "Displayed Link",

                  "Document", "Section", "Item", "Cited Guidance", "Guidance Section", "Release Notes",

                  "Author", "Question(s) for FDA", "Answer of the FDA", "Authoring section",

                  "Comments/questions for discussion", "Response Lex"
                  ]

sdtmf = ensure_columns(sdtmig, target_columns)
adamf = ensure_columns(adamig, target_columns)
fdaf = ensure_columns(fdabr, target_columns)
defsf = ensure_columns(defs, target_columns)
defcf = ensure_columns(defc, target_columns)

sdtmf = sdtmf[target_columns].copy()
adamf = adamf[target_columns].copy()
fdaf = fdaf[target_columns].copy()
defsf = defsf[target_columns].copy()
defcf = defcf[target_columns].copy()

concat_sot = pd.concat([sdtmf, adamf, fdaf, defsf, defcf])
print(concat_sot.shape)

(1880, 38)


In [14]:
concat_sot.head(25)

,conformance_id,Rule ID,cg_version,Rule Version,Class,Subclass,Domain,Variable,Define-XML Element,Scope Section,...,Item,Cited Guidance,Guidance Section,Release Notes,Author,Question(s) for FDA,Answer of the FDA,Authoring section,Comments/questions for discussion,Response Lex
2,CG0001,CG0001,,1,ALL,,ALL,DOMAIN,,,...,NaN,Using SDTM-specified standard domain names and...,,Section and cited guidance updated. Orginal c...,,,,,,
5,CG0002,CG0002,,1,ALL,,ALL,--DUR,,,...,--DUR,Used only if collected on the CRF and not deri...,,Cited guidance updated. Reference to model sec...,,,,,,
8,CG0006,CG0006,,2,ALL,,ALL,--DY,,,...,NaN,The Study Day value is incremented by 1 for ea...,,No Change,,,,,,
11,CG0007,CG0007,,1,ALL,,ALL,--DY,,,...,NaN,"The permissible Study Day variables (--DY, --S...",,No Change,,,,,,
14,CG0008,CG0008,,1,ALL,,ALL,--ELTM,,,...,--ELTM,The interval of time between a planned time po...,,Cited guidance updated. Reference to model sec...,,,,,,
17,CG0009,CG0009,,1,ALL,,ALL,EPOCH,,,...,"DS, Assumption 4.c",EPOCH may be included as a timing variable as ...,,No Change,,,,,,
20,CG0010,CG0010,,2,ALL,,ALL,GEN,,,...,NaN,Each variable can be classified according to i...,,"""Metadata attribute of"" added into the rule.",,,,,,
23,CG0011,CG0011,,1,ALL,,ALL,GEN,,,...,NaN,Following SDTM-specified controlled terminolog...,,No Change,,,,,,
26,CG0012,CG0012,,1,ALL,,ALL,GEN,,,...,NaN,IG v3.4[3.2.2][Using SDTM-specified data types...,,"""The SDTM describes the name, label, role, and...",,,,,,
29,CG0013,CG0013,,1,ALL,,ALL,GEN,,,...,NaN,IG v3.4[2.5][Sponsors may not add any variable...,,IG section number updated and citation clarifi...,,,,,,


## Read in Rule Collation with additional metadata

From here we get rule type and implementation details like check_operators and operations. Only available, it seems, for SDTM IG conformance_ids, but better than nothing.

We do *not* get the CORE ID from here, that's from the Rule Editor dump.

In [15]:
ruc = pd.read_excel(f"{dump_version}-SOT-sources/Collation of Rule Editor Entries.xlsx", sheet_name='Rule Editor')
ruc.columns = ruc.columns.str.replace(r'\s+', ' ', regex=True).str.strip()
print(ruc.shape)
ruc.head(5)

(432, 22)


,Rule Id,Core Id,Core Version,Authority Organization,Standards Name,Standards Version,Rule Identifier Id,Rule Identifier Rule Version,Rule Identifier Conformance Guide Version,Description,...,Rule Type,Check,Match Datasets,Operations,Outcome Message,Outcome Output Variables,Citations,References Origin,Sensitivity,Severity
0,CG0001,CDISC.SDTMIG.CG0001,1,CDISC,SDTMIG,3.4,CG0001,1,2,DOMAIN name should use a SDTM-specified standa...,...,Range & Limit,all:\n - name: DOMAIN\n operator: uses_valid_c...,NaN,NaN,DOMAIN Code is not a published DOMAIN Code in ...,NaN,- Cited Guidance: Using SDTM-specified standar...,SDTM and SDTMIG Conformance Rules,Record,Notice
1,CG0002,CDISC.SDTMIG.CG0002,1,CDISC,SDTMIG,3.4,CG0002,1,2,Raise a message when --DUR exists in a dataset...,...,Variable Presence,all:\n - name: --DUR\n operator: exists\n,NaN,NaN,"--DUR exists in a dataset, confirm collected a...",NaN,- Cited Guidance: Used only if collected on th...,SDTM and SDTMIG Conformance Rules,Dataset,Warning
2,CG0006,CDISC.SDTMIG.CG0006,1,CDISC,SDTMIG,3.4,CG0006,2,2,Raise an error when --DY is not calculated as ...,...,Value Presence,all:\n - name: --DY\n operator: non_empty\n - ...,- Name: DM\n Keys:\n - USUBJID\n,- name: --DTC\n operator: dy # does dy op take...,--DY is not calculated correctly even though t...,- --DY - --DTC - RFSTDTC,- Cited Guidance: The Study Day value is incre...,SDTM and SDTMIG Conformance Rules,Record,Error
3,CG0007,CDISC.SDTMIG.CG0007,1,CDISC,SDTMIG,3.4,CG0007,1,2,Raise an error when the date portion of --DTC ...,...,Value Presence,all:\n - name: --DY\n operator: non_empty\n - ...,- Name: DM\n Keys:\n - USUBJID\n,NaN,The date portion of --DTC is not complete date...,- --DY - --DTC - RFSTDTC,- Cited Guidance: The permissible Study Day va...,SDTM and SDTMIG Conformance Rules,Record,Error
4,CG0008,CDISC.SDTMIG.CG0008,1,CDISC,SDTMIG,3.4,CG0008,1,2,"Raise an error when --TPTREF is empty, but --E...",...,Value Presence,all:\n - name: --TPTREF\n operator: empty\n - ...,NaN,NaN,--TPTREF is empty and --ELTM is not empty,- --TPTREF - --ELTM,- Cited Guidance: The interval of time between...,SDTM and SDTMIG Conformance Rules,Record,Error


In [16]:
print(ruc.columns)
# ensure we don't explode the SOT dataframe post merge:
assert sum(ruc.duplicated("Rule Id")) == 0

# select columns and rename
rucf = ruc[[
    "Rule Id", "Description", "Scopes Classes", "Scopes Domains",
    "Rule Type", "Check", "Match Datasets", "Operations", "Outcome Message",
    "Outcome Output Variables", "Sensitivity", "Severity"
    ]].copy()
rucf_convert_dict = {
    "Rule Id": "conformance_id",
    "Description": "impl_description",
    "Scopes Classes": "impl_scope_classes",
    "Scopes Domains": "impl_scope_domains",
    "Rule Type": "rule_type",
    "Check": "impl_check_operators",
    "Match Datasets": "impl_match_datasets",
    "Operations": "impl_operations",
    "Outcome Message": "impl_error_message",
    "Outcome Output Variables": "impl_outcome_variables",
    "Sensitivity": "sensitivity",
    "Severity": "severity"
    }
rucf = rucf.rename(columns=rucf_convert_dict)
rucf.head(15)

Index(['Rule Id', 'Core Id', 'Core Version', 'Authority Organization',
       'Standards Name', 'Standards Version', 'Rule Identifier Id',
       'Rule Identifier Rule Version',
       'Rule Identifier Conformance Guide Version', 'Description',
       'Scopes Classes', 'Scopes Domains', 'Rule Type', 'Check',
       'Match Datasets', 'Operations', 'Outcome Message',
       'Outcome Output Variables', 'Citations', 'References Origin',
       'Sensitivity', 'Severity'],
      dtype='object')


,conformance_id,impl_description,impl_scope_classes,impl_scope_domains,rule_type,impl_check_operators,impl_match_datasets,impl_operations,impl_error_message,impl_outcome_variables,sensitivity,severity
0,CG0001,DOMAIN name should use a SDTM-specified standa...,\n Include:\n - All\n,Domains:\n Include:\n - All,Range & Limit,all:\n - name: DOMAIN\n operator: uses_valid_c...,NaN,NaN,DOMAIN Code is not a published DOMAIN Code in ...,NaN,Record,Notice
1,CG0002,Raise a message when --DUR exists in a dataset...,\n Include:\n - All\n,Domains:\n Include:\n - All,Variable Presence,all:\n - name: --DUR\n operator: exists\n,NaN,NaN,"--DUR exists in a dataset, confirm collected a...",NaN,Dataset,Warning
2,CG0006,Raise an error when --DY is not calculated as ...,\n Include:\n - All\n,Domains:\n Include:\n - All,Value Presence,all:\n - name: --DY\n operator: non_empty\n - ...,- Name: DM\n Keys:\n - USUBJID\n,- name: --DTC\n operator: dy # does dy op take...,--DY is not calculated correctly even though t...,- --DY - --DTC - RFSTDTC,Record,Error
3,CG0007,Raise an error when the date portion of --DTC ...,\n Include:\n - All\n,Domains:\n Include:\n - All,Value Presence,all:\n - name: --DY\n operator: non_empty\n - ...,- Name: DM\n Keys:\n - USUBJID\n,NaN,The date portion of --DTC is not complete date...,- --DY - --DTC - RFSTDTC,Record,Error
4,CG0008,"Raise an error when --TPTREF is empty, but --E...",\n Include:\n - All\n,Domains:\n Include:\n - All,Value Presence,all:\n - name: --TPTREF\n operator: empty\n - ...,NaN,NaN,--TPTREF is empty and --ELTM is not empty,- --TPTREF - --ELTM,Record,Error
5,CG0009,Raise an error when EPOCH is not in TA.EPOCH,\n Include:\n - All\n,Domains:\n Include:\n - All,Range & Limit,all:\n - name: EPOCH\n operator: is_not_contai...,NaN,- domain: TA\n id: $ta_epoch\n name: EPOCH\n o...,EPOCH is not in TA.EPOCH,- EPOCH,Record,Error
6,CG0010,Trigger warning when metadata attribute of var...,\n Include:\n - All\n,Domains:\n Include:\n - All,Variable Metadata Check,all:\n - name: null\n operator: not_equal_to\n...,NaN,NaN,Metadata attribute of variable role does not e...,NaN,Dataset,Warning
7,CG0011,Trigger error when variable format does not ma...,\n Include:\n - All\n,Domains:\n Include:\n - All,Range & Limit,all:\n - name: null\n operator: not_equal_to\n...,NaN,NaN,Variable format does not match SDTM-specified ...,NaN,Dataset,Warning
8,CG0012,Trigger error when variable type does not matc...,\n Include:\n - All\n,Domains:\n Include:\n - All,Variable Metadata Check against Define XML,all:\n - name: null\n operator: null\n - name:...,NaN,NaN,Variable type does not match IG Type or Model ...,NaN,Dataset,Error
9,CG0013,Trigger error when variable is not an allowed ...,\n Include:\n - All\n,Domains:\n Include:\n - All,Variable Metadata Check against Define XML,all:\n - name: variable_name\n operator: not_e...,NaN,NaN,Variable is not an allowed variable for an Obs...,NaN,Record,Error


## Merge metadata to SOT

In [17]:
enriched_sot = concat_sot.merge(rucf, on="conformance_id", how="left")
assert concat_sot.shape[0] == enriched_sot.shape[0]
print(f"{concat_sot.shape} = {enriched_sot.shape}")
enriched_sot.head(15)

(1880, 38) = (1880, 49)


,conformance_id,Rule ID,cg_version,Rule Version,Class,Subclass,Domain,Variable,Define-XML Element,Scope Section,...,impl_scope_classes,impl_scope_domains,rule_type,impl_check_operators,impl_match_datasets,impl_operations,impl_error_message,impl_outcome_variables,sensitivity,severity
0,CG0001,CG0001,,1,ALL,,ALL,DOMAIN,,,...,\n Include:\n - All\n,Domains:\n Include:\n - All,Range & Limit,all:\n - name: DOMAIN\n operator: uses_valid_c...,NaN,NaN,DOMAIN Code is not a published DOMAIN Code in ...,NaN,Record,Notice
1,CG0002,CG0002,,1,ALL,,ALL,--DUR,,,...,\n Include:\n - All\n,Domains:\n Include:\n - All,Variable Presence,all:\n - name: --DUR\n operator: exists\n,NaN,NaN,"--DUR exists in a dataset, confirm collected a...",NaN,Dataset,Warning
2,CG0006,CG0006,,2,ALL,,ALL,--DY,,,...,\n Include:\n - All\n,Domains:\n Include:\n - All,Value Presence,all:\n - name: --DY\n operator: non_empty\n - ...,- Name: DM\n Keys:\n - USUBJID\n,- name: --DTC\n operator: dy # does dy op take...,--DY is not calculated correctly even though t...,- --DY - --DTC - RFSTDTC,Record,Error
3,CG0007,CG0007,,1,ALL,,ALL,--DY,,,...,\n Include:\n - All\n,Domains:\n Include:\n - All,Value Presence,all:\n - name: --DY\n operator: non_empty\n - ...,- Name: DM\n Keys:\n - USUBJID\n,NaN,The date portion of --DTC is not complete date...,- --DY - --DTC - RFSTDTC,Record,Error
4,CG0008,CG0008,,1,ALL,,ALL,--ELTM,,,...,\n Include:\n - All\n,Domains:\n Include:\n - All,Value Presence,all:\n - name: --TPTREF\n operator: empty\n - ...,NaN,NaN,--TPTREF is empty and --ELTM is not empty,- --TPTREF - --ELTM,Record,Error
5,CG0009,CG0009,,1,ALL,,ALL,EPOCH,,,...,\n Include:\n - All\n,Domains:\n Include:\n - All,Range & Limit,all:\n - name: EPOCH\n operator: is_not_contai...,NaN,- domain: TA\n id: $ta_epoch\n name: EPOCH\n o...,EPOCH is not in TA.EPOCH,- EPOCH,Record,Error
6,CG0010,CG0010,,2,ALL,,ALL,GEN,,,...,\n Include:\n - All\n,Domains:\n Include:\n - All,Variable Metadata Check,all:\n - name: null\n operator: not_equal_to\n...,NaN,NaN,Metadata attribute of variable role does not e...,NaN,Dataset,Warning
7,CG0011,CG0011,,1,ALL,,ALL,GEN,,,...,\n Include:\n - All\n,Domains:\n Include:\n - All,Range & Limit,all:\n - name: null\n operator: not_equal_to\n...,NaN,NaN,Variable format does not match SDTM-specified ...,NaN,Dataset,Warning
8,CG0012,CG0012,,1,ALL,,ALL,GEN,,,...,\n Include:\n - All\n,Domains:\n Include:\n - All,Variable Metadata Check against Define XML,all:\n - name: null\n operator: null\n - name:...,NaN,NaN,Variable type does not match IG Type or Model ...,NaN,Dataset,Error
9,CG0013,CG0013,,1,ALL,,ALL,GEN,,,...,\n Include:\n - All\n,Domains:\n Include:\n - All,Variable Metadata Check against Define XML,all:\n - name: variable_name\n operator: not_e...,NaN,NaN,Variable is not an allowed variable for an Obs...,NaN,Record,Error


## Rule Editor dump

We need this to get the latest CORE-id assignments and to create the true 1-n mapping from conformance_id to core_id.

empty core_ids are going to get an assignment here (conformance_id + "-n") for the rest of the workflow to make sense.

In [18]:
rer = pd.read_csv(f"{dump_version}-SOT-sources/RuleEditorRules_20251209.csv")
rer.head(5)

,Core-ID,CDISC Rule ID,Error Message,Description,Standard Name,Standard Version,Scope,Executability,Status
0,NaN,CG0011C2,"The variable is under controlled terminology, ...",Variables that are under CT must have an assoc...,SDTMIG,"3.2, 3.3, 3.4","{""Classes"":{""Include"":[""ALL""]},""Domains"":{""Inc...",Fully Executable,Draft
1,CORE-000201,"CG0029, TIG0311, SEND109, TIG0046",USUBJID is not found in DM.USUBJID,Trigger error when domain is not an AP-- domai...,"SDTMIG, TIG, SENDIG, SENDIG-DART, SENDIG-GENETOX","3.4, 3.3, 3.2, 1.0, 3.0, 3.1, 3.1.1, 1.1, 1.2","{""Classes"":{""Include"":[""ALL""]},""Domains"":{""Exc...",Fully Executable,Published
2,CDISC.SDTMIG.CG0562,"CG0562, TIG0648",'--REPNUM is null or is not unique per subject...,Raise an error when REPNUM is in the dataset a...,"SDTMIG, TIG","3.4, 3.3, 1.0","{""Classes"":{""Include"":[""FINDINGS""]},""Domains"":...",Fully Executable,Draft
3,CORE-000956,DDF00174,More than 1 study identifier is specified for ...,An identified organization is not expected to ...,USDM,4.0,"{""Entities"":{""Include"":[""StudyIdentifier""]}}",Partially Executable,Published
4,CORE-000955,DDF00173,The identifier text is not unique within the s...,Every identifier must be unique within the sco...,USDM,4.0,"{""Entities"":{""Include"":[""StudyIdentifier"",""Ref...",Fully Executable,Published


In [19]:
# split up to only retrieve our standards of interest:
rer['std'] = rer['Standard Name'].apply(lambda x: x.split(', ') if isinstance(x, str) else None)
rer['rids'] = rer['CDISC Rule ID'].apply(lambda x: x.split(', ') if isinstance(x, str) else None)

unique_strings = list(pd.unique(rer['std'].explode().dropna()))
print(unique_strings)

# BEWARE - THIS IS A MANUALLY CREATED LIST FROM THE OUPUTS OF THE NEXT CELL
relevant_stds = {'SDTMIG', 'ADaMIG', 'Internal Standard', 'ADaMIG-MD', 'ADTTE', 'ADAMIG', 'Define-XML'}
print(relevant_stds)

# now filter to relevant standards
mask = rer["std"].apply(
    lambda x: bool(relevant_stds.intersection(x)) if isinstance(x, (list, set, tuple)) else False
)

rerf = rer[mask].copy()
print(rerf.shape)
rerf.head(10)

['SDTMIG', 'TIG', 'SENDIG', 'SENDIG-DART', 'SENDIG-GENETOX', 'USDM', 'SENDIG-AR', 'ADaMIG', 'ADaMIG-MD', 'ADTTE', 'OCCDS', 'ADAMIG', 'Model v2.1', 'Define-XML', 'SENDID-GENETOX']
{'ADAMIG', 'SDTMIG', 'ADaMIG', 'Define-XML', 'ADTTE', 'Internal Standard', 'ADaMIG-MD'}
(1262, 11)


,Core-ID,CDISC Rule ID,Error Message,Description,Standard Name,Standard Version,Scope,Executability,Status,std,rids
0,NaN,CG0011C2,"The variable is under controlled terminology, ...",Variables that are under CT must have an assoc...,SDTMIG,"3.2, 3.3, 3.4","{""Classes"":{""Include"":[""ALL""]},""Domains"":{""Inc...",Fully Executable,Draft,[SDTMIG],[CG0011C2]
1,CORE-000201,"CG0029, TIG0311, SEND109, TIG0046",USUBJID is not found in DM.USUBJID,Trigger error when domain is not an AP-- domai...,"SDTMIG, TIG, SENDIG, SENDIG-DART, SENDIG-GENETOX","3.4, 3.3, 3.2, 1.0, 3.0, 3.1, 3.1.1, 1.1, 1.2","{""Classes"":{""Include"":[""ALL""]},""Domains"":{""Exc...",Fully Executable,Published,"[SDTMIG, TIG, SENDIG, SENDIG-DART, SENDIG-GENE...","[CG0029, TIG0311, SEND109, TIG0046]"
2,CDISC.SDTMIG.CG0562,"CG0562, TIG0648",'--REPNUM is null or is not unique per subject...,Raise an error when REPNUM is in the dataset a...,"SDTMIG, TIG","3.4, 3.3, 1.0","{""Classes"":{""Include"":[""FINDINGS""]},""Domains"":...",Fully Executable,Draft,"[SDTMIG, TIG]","[CG0562, TIG0648]"
19,FDA.SDTMIG.SD1149,SD1149,Expected variable with missing value for all r...,Expected variables may contain some null value...,"SDTMIG, SENDIG, SENDIG-AR, SENDIG-DART","3.1.2, 3.1.3, 3.2, 3.3, 3.0, 3.1, 3.1.1, 1.0, 1.1","{""Classes"":{""Include"":[""EVENTS"",""FINDINGS"",""FI...",Fully Executable,Draft,"[SDTMIG, SENDIG, SENDIG-AR, SENDIG-DART]",[SD1149]
50,NaN,CG0011B1,NaN,NaN,SDTMIG,"3.2, 3.3, 3.4","{""Classes"":{""Include"":[""ALL""]},""Domains"":{""Inc...",Fully Executable,Draft,[SDTMIG],[CG0011B1]
51,NaN,CG0011A1,Variable value for --DUR does not comply with ...,NaN,SDTMIG,"3.2, 3.3, 3.4","{""Classes"":{""Include"":[""ALL""]},""Domains"":{""Inc...",Fully Executable,Draft,[SDTMIG],[CG0011A1]
53,NaN,CG0011A2,NaN,NaN,SDTMIG,"3.2, 3.3, 3.4","{""Classes"":{""Include"":[""ALL""]},""Domains"":{""Inc...",Fully Executable,Draft,[SDTMIG],[CG0011A2]
54,CDISC.SDTMIG.CG0019,CG0019,Records are not unique as per sponsor defined ...,Trigger error if records are not unique as per...,SDTMIG,"3.4, 3.2, 3.3","{""Classes"":{""Include"":[""ALL""]},""Domains"":{""Inc...",Fully Executable,Draft,[SDTMIG],[CG0019]
55,FDA.SDTMIG.SD1097,FB0101,The treatment-emergent flag is missing.,Treatment-emergent flag should be present in S...,SDTMIG,"3.2, 3.3, 3.4","{""Classes"":{""Include"":[""ALL""]},""Domains"":{""Inc...",Fully Executable,Draft,[SDTMIG],[FB0101]
79,CORE-000953,"CG0370, TIG0534",Value for IDVAR in CO does not represent a var...,Part C - Raise and error when IDVAR in CO is n...,"SDTMIG, TIG","3.4, 3.2, 3.3, 1.0","{""Classes"":{""Include"":[""SPECIAL PURPOSE""]},""Do...",Fully Executable,Published,"[SDTMIG, TIG]","[CG0370, TIG0534]"


In [20]:
# filter all available rules down to the ones where the IDs are part of our rulesets of interest

allowed_patterns = [
    r"^CG.*$",                   # SDTM Rules
    r"^FB.*",                    # FDA Business Rules
    r"^\d+$",                    # ADAM & Define Rules
]
allowed_regexes = [re.compile(p) for p in allowed_patterns]

def keep_allowed(id_list):
    if not isinstance(id_list, (list, tuple)):
        return []
    return [
        s for s in id_list
        if any(regex.search(s) for regex in allowed_regexes)
    ]

rerf["rid"] = rerf["rids"].apply(keep_allowed)

# now drop all rules that don't have at least one ID of interest
print(rerf.shape)
rerf = rerf[rerf["rid"].apply(len) > 0].copy()
print(rerf.shape)

# diagnostic: number of rules with multiple CG quoted:
rerf[rerf["rid"].apply(len) > 1][["Core-ID", "rids", 'rid']].head(50)

#rerf.head(15)

(1262, 12)
(772, 12)


,Core-ID,rids,rid
82,CDISC.SDTMIG.CG0013-1,"[CG0013, CG0351]","[CG0013, CG0351]"
83,CORE-000550,"[CG0013, TIG0298, CG0351]","[CG0013, CG0351]"
151,CORE-000251,"[CG0067, FB0614]","[CG0067, FB0614]"
190,CORE-000254,"[CG0134, TIG0383, FB0603]","[CG0134, FB0603]"
267,CORE-000105,"[CG0569, TIG0653, FB2602]","[CG0569, FB2602]"
306,CORE-000852,"[CG0330, CG0664, TIG0698, SEND48]","[CG0330, CG0664]"
327,CORE-000259,"[CG0171, FB0613]","[CG0171, FB0613]"
332,CORE-000529,"[CG0006, TIG0291, SEND73, TIG0274, SEND74, TIG...","[CG0006, FB1603]"
351,CORE-000368,"[CG0120, CG0121]","[CG0120, CG0121]"
353,CORE-000367,"[CG0117, CG0118]","[CG0117, CG0118]"


In [21]:
# note that we have several rules that refer to several conformance_ids, so there are two views on this source of truth (because of n-n mapping):
# 1: have a conformance_id view where conformance_id = unique row, so that all corresponding engine rules are summarized into a list
#    (overall progress view, where progress has to be manually defined, bc the judgement of whether a CG is covered can only be made manually and requires a field)
# 2: have a rule_id view, where rule_id = unique row, so that all corresponding conformance_ids are summarized into a list
#    (contributor repository and rule progress overview, which can be merged with regression)

# explode rid column, create a cleaned_rid column, then impute ADAM, DES, and DEC IDs
print(rerf.shape)
rerf["conformance_id"] = rerf["rid"]
rerfe = rerf.explode("conformance_id").copy()
print(rerfe.shape)
rerfe["conformance_id"] = rerfe["conformance_id"].str[:6]

# fix ADAM and DEFS IDs, until CDISC cleans it up
adam_mask = rerfe["std"].apply(lambda lst: ("ADAMIG" in lst) or ("ADaMIG" in lst) or ("ADaMIG-MD" in lst)or ("ADTTE" in lst))
defs_mask = rerfe["std"].apply(lambda lst: "Define-XML" in lst)

rerfe.loc[adam_mask, "conformance_id"] = "AD" + rerfe.loc[adam_mask, "conformance_id"].apply(lambda x: f"{int(x):04d}")
rerfe.loc[defs_mask, "conformance_id"] = "DS" + rerfe.loc[defs_mask, "conformance_id"].apply(lambda x: f"{int(x):04d}")

# duplicates analysis (rules with 1-n mapping)
dupes = rerfe["conformance_id"].value_counts()
print(f'duplicate conformance_ids: {dupes[dupes > 1].index.tolist()}')

rerfe.head(50)
#rerfe[rerfe["rid"].apply(len) > 1][["Core-ID", "rids", 'rid', 'conformance_id']].head(500)


(772, 12)
(809, 13)
duplicate conformance_ids: ['CG0011', 'AD0364', 'CG0370', 'CG0531', 'CG0013', 'CG0351', 'CG0430', 'CG0429', 'CG0014', 'AD0254', 'AD0252', 'CG0372']


,Core-ID,CDISC Rule ID,Error Message,Description,Standard Name,Standard Version,Scope,Executability,Status,std,rids,rid,conformance_id
0,NaN,CG0011C2,"The variable is under controlled terminology, ...",Variables that are under CT must have an assoc...,SDTMIG,"3.2, 3.3, 3.4","{""Classes"":{""Include"":[""ALL""]},""Domains"":{""Inc...",Fully Executable,Draft,[SDTMIG],[CG0011C2],[CG0011C2],CG0011
1,CORE-000201,"CG0029, TIG0311, SEND109, TIG0046",USUBJID is not found in DM.USUBJID,Trigger error when domain is not an AP-- domai...,"SDTMIG, TIG, SENDIG, SENDIG-DART, SENDIG-GENETOX","3.4, 3.3, 3.2, 1.0, 3.0, 3.1, 3.1.1, 1.1, 1.2","{""Classes"":{""Include"":[""ALL""]},""Domains"":{""Exc...",Fully Executable,Published,"[SDTMIG, TIG, SENDIG, SENDIG-DART, SENDIG-GENE...","[CG0029, TIG0311, SEND109, TIG0046]",[CG0029],CG0029
2,CDISC.SDTMIG.CG0562,"CG0562, TIG0648",'--REPNUM is null or is not unique per subject...,Raise an error when REPNUM is in the dataset a...,"SDTMIG, TIG","3.4, 3.3, 1.0","{""Classes"":{""Include"":[""FINDINGS""]},""Domains"":...",Fully Executable,Draft,"[SDTMIG, TIG]","[CG0562, TIG0648]",[CG0562],CG0562
50,NaN,CG0011B1,NaN,NaN,SDTMIG,"3.2, 3.3, 3.4","{""Classes"":{""Include"":[""ALL""]},""Domains"":{""Inc...",Fully Executable,Draft,[SDTMIG],[CG0011B1],[CG0011B1],CG0011
51,NaN,CG0011A1,Variable value for --DUR does not comply with ...,NaN,SDTMIG,"3.2, 3.3, 3.4","{""Classes"":{""Include"":[""ALL""]},""Domains"":{""Inc...",Fully Executable,Draft,[SDTMIG],[CG0011A1],[CG0011A1],CG0011
53,NaN,CG0011A2,NaN,NaN,SDTMIG,"3.2, 3.3, 3.4","{""Classes"":{""Include"":[""ALL""]},""Domains"":{""Inc...",Fully Executable,Draft,[SDTMIG],[CG0011A2],[CG0011A2],CG0011
54,CDISC.SDTMIG.CG0019,CG0019,Records are not unique as per sponsor defined ...,Trigger error if records are not unique as per...,SDTMIG,"3.4, 3.2, 3.3","{""Classes"":{""Include"":[""ALL""]},""Domains"":{""Inc...",Fully Executable,Draft,[SDTMIG],[CG0019],[CG0019],CG0019
55,FDA.SDTMIG.SD1097,FB0101,The treatment-emergent flag is missing.,Treatment-emergent flag should be present in S...,SDTMIG,"3.2, 3.3, 3.4","{""Classes"":{""Include"":[""ALL""]},""Domains"":{""Inc...",Fully Executable,Draft,[SDTMIG],[FB0101],[FB0101],FB0101
79,CORE-000953,"CG0370, TIG0534",Value for IDVAR in CO does not represent a var...,Part C - Raise and error when IDVAR in CO is n...,"SDTMIG, TIG","3.4, 3.2, 3.3, 1.0","{""Classes"":{""Include"":[""SPECIAL PURPOSE""]},""Do...",Fully Executable,Published,"[SDTMIG, TIG]","[CG0370, TIG0534]",[CG0370],CG0370
80,CORE-000916,"CG0370, TIG0534",Value for IDVAR in RELREC does not represent a...,Part B - Raise and error when IDVAR in RELREC ...,"SDTMIG, TIG","3.4, 3.2, 3.3, 1.0","{""Classes"":{""Include"":[""RELATIONSHIP""]},""Domai...",Fully Executable,Published,"[SDTMIG, TIG]","[CG0370, TIG0534]",[CG0370],CG0370


In [22]:
# create CG-centric view:
# merge based on the column with SOT, group-by conformance_id (check: should have same length as sot df)
rerfef = rerfe[["Core-ID", "conformance_id"]].copy()

gc_centric = enriched_sot.merge(rerfef, on="conformance_id", how="left")
gc_centric_out = enriched_sot.merge(rerfef, on="conformance_id", how="outer")

if not gc_centric.shape[0] == gc_centric_out.shape[0]:
    print("####### JOIN ANALYSIS #######")
    print(f"difference gc_centric: {gc_centric[~gc_centric.apply(tuple, axis=1).isin(gc_centric_out.apply(tuple, axis=1))][["conformance_id", "Core-ID"]]}")
    print(f"difference gc_centric_out: {gc_centric_out[~gc_centric_out.apply(tuple, axis=1).isin(gc_centric.apply(tuple, axis=1))][["conformance_id", "Core-ID"]]}")
    print("NOTE: you can safely ignore CG0999, as this is a template used by CDISC (see rule editor)")
    print("####### JOIN ANALYSIS END #######")
    print()

print(enriched_sot.shape)
print(gc_centric.shape)
gc_centric = gc_centric.groupby("conformance_id").agg({"Core-ID": lambda s: list(s.dropna()), **{col: "first" for col in gc_centric.columns if col not in ["Core-ID"]}}).copy()
gc_centric["Core-ID"] = gc_centric["Core-ID"].apply(lambda lst: "; ".join(lst))
print(gc_centric.shape)

# write out to file
gc_centric.to_csv("GC_centric_SOT.csv", header=True, index=False, sep=",", quotechar='"')

####### JOIN ANALYSIS #######
difference gc_centric: Empty DataFrame
Columns: [conformance_id, Core-ID]
Index: []
difference gc_centric_out:      conformance_id              Core-ID
1902         CG0999  CDISC.SDTMIG.CG0999
NOTE: you can safely ignore CG0999, as this is a template used by CDISC (see rule editor)
####### JOIN ANALYSIS END #######

(1880, 49)
(1902, 50)
(1880, 50)


In [23]:
# create rule-centric view:
# merge in SOT conformance IDs and group-by rule_id
# based on these rule-ids, we then need to create the repository and merge in regression

# move conformance_id column to second place
conf_col = rerfe.pop("conformance_id")
rerfe.insert(1, "conformance_id", conf_col)
re_sot = rerfe.copy()


# aggregate on Core-ID for the rule-centric view
re_sot = re_sot.groupby("Core-ID").agg({"conformance_id": lambda s: list(set(list(s.dropna()))), **{col: "first" for col in re_sot.columns if col not in ["conformance_id"]}}).copy()
re_sot["conformance_id"] = re_sot["conformance_id"].apply(lambda lst: "; ".join(lst))

# sort
re_sot = rerfe.sort_values(by=["conformance_id", "Core-ID"])

# write out to file
re_sot.to_csv("Rule_centric_SOT.csv", header=True, index=False, sep=",", quotechar='"')
re_sot.head(500)

,Core-ID,conformance_id,CDISC Rule ID,Error Message,Description,Standard Name,Standard Version,Scope,Executability,Status,std,rids,rid
1033,CORE-000560,AD0001,1,Dataset ADSL does not exist,The ADSL dataset must exist,ADaMIG,"1.3, 1.2, 1.1, 1.0","{""Domains"":{""Include"":[""ALL""]},""Data_Structure...",Fully Executable,Published,[ADaMIG],[1],[1]
365,CDISC.ADaMIG.AD0005,AD0005,5,Variable with a suffix of FL does not has a va...,A variable with a suffix of FL must have a val...,ADaMIG,"1.3, 1.2, 1.1, 1.0","{""Classes"":{""Include"":[""ALL""]},""Subclasses"":{""...",Fully Executable,Draft,[ADaMIG],[5],[5]
943,CDISC.ADaMIG.AD0006,AD0006,6,Variable with a suffix of FN does must have th...,'A variable with a suffix of FN must have a va...,ADaMIG,"1.3, 1.2, 1.1, 1.0","{""Subclasses"":{""Include"":[""ALL""]},""Data_Struct...",Fully Executable,Draft,[ADaMIG],[6],[6]
337,CDISC.ADaMIG.AD0007,AD0007,7,Variable with a suffix of FN does must have th...,'A variable with a suffix of FN must have a va...,ADaMIG,"1.3, 1.2, 1.1, 1.0","{""Subclasses"":{""Include"":[""ALL""]},""Data_Struct...",Fully Executable,Draft,[ADaMIG],[7],[7]
1044,CDSIC.ADaMIG.AD0010,AD0010,10,A variable with a suffix of FL is equal to Y a...,'A variable with a suffix of FL is equal to Y ...,ADaMIG,"1.3, 1.2, 1.1, 1.0","{""Classes"":{""Include"":[""ALL""]},""Data_Structure...",Fully Executable,Draft,[ADaMIG],[10],[10]
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1925,CDISC.SDTMIG.CG0418,CG0418,CG0418,NaN,NaN,SDTMIG,"3.2, 3.3, 3.4","{""Classes"":{""Include"":[""ALL""]},""Domains"":{""Inc...",Fully Executable,Draft,[SDTMIG],[CG0418],[CG0418]
938,CORE-000202,CG0419,"CG0419, TIG0574",RELTYPE is populated when IDVAR is populated w...,"When IDVAR is populated with a --SEQ value, RE...","SDTMIG, TIG","3.4, 3.2, 3.3, 1.0","{""Classes"":{""Include"":[""RELATIONSHIP""]},""Domai...",Fully Executable,Published,"[SDTMIG, TIG]","[CG0419, TIG0574]",[CG0419]
598,CORE-000240,CG0420,"CG0420, TIG0575","'--STRF is populated when --OCCUR = ""N""","When --OCCUR = 'N', --STRF must not be populated","SDTMIG, TIG","3.4, 3.2, 3.3, 1.0","{""Classes"":{""Include"":[""INTERVENTIONS"",""EVENTS...",Fully Executable,Published,"[SDTMIG, TIG]","[CG0420, TIG0575]",[CG0420]
582,CORE-000241,CG0421,"CG0421, TIG0576","'--ENRF is populated when --OCCUR = ""N""","When --OCCUR = 'N', --ENRF must not be populated","SDTMIG, TIG","3.4, 3.2, 3.3, 1.0","{""Classes"":{""Include"":[""INTERVENTIONS"",""EVENTS...",Fully Executable,Published,"[SDTMIG, TIG]","[CG0421, TIG0576]",[CG0421]


# CDISC ToDo/Align

1) need unique IDs for rules, even when in dev, ideally these are composed of the below with a -XX addendum
2) within standard conformance rule IDs need to be unique (can't have 1-800 for both adam and define, so we propose: ADXXXX for adam, DSXXXX for define-xml schema, and DCXXXX for define-xml cross-checks)
3) process to agree on: - we can only mark a CG as complete manually, bc there could be 1-n core rules required for execution
4) repo is structured by those core rules, and we need a process to update folder structures and rule unique IDs (point 1 and when CORE-ID is assigned)

## Add regression columns for overview --> this needs to be moved into regression
